### REFERENCE SOLUTION — Diagnostic EDA on retail_transactions_raw.csv
### Try your own diagnosis first. Use this only to check your work / get unstuck.
 
**Structure follows a standard "diagnose before you clean" EDA pipeline:**
  1. Structural overview
  2. Missingness audit
  3. Duplicate audit
  4. Type / format audit (per column)
  5. Range & domain-validity audit (outliers, impossible values)
  6. Referential integrity audit (vs customers_reference.csv)
  7. Summary report of every issue found

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("retail_transactions_raw.csv")
customers = pd.read_csv("customers_reference.csv")

In [3]:
issues = []  # collect (column, issue_description, count) for the final report
 
print("="*70)
print("1. STRUCTURAL OVERVIEW")
print("="*70)
print(df.shape)
print(df.dtypes)
print(df.head(3))

1. STRUCTURAL OVERVIEW
(879, 15)
transaction_id      float64
customer_id         float64
customer_name           str
email                   str
transaction_date        str
product_category        str
quantity            float64
unit_price              str
discount_pct        float64
payment_method          str
country                 str
shipping_weight         str
rating              float64
is_returned             str
notes                   str
dtype: object
   transaction_id  customer_id customer_name                   email  \
0          5046.0       1159.0  Alina Farooq  alina.farooq@gmail.com   
1          5296.0       1077.0  Nimra Sheikh  nimra.sheikh@gmail.com   
2          5789.0       1005.0     Sara Khan     sara.khan@gmail.com   

  transaction_date product_category  quantity unit_price  discount_pct  \
0              NaN           BEAUTY       7.0    1674.76          49.8   
1       07/04/2025      electronics       6.0     2411.4          54.8   
2      21 Nov 2027    

In [4]:
print("\n" + "="*70)
print("2. MISSINGNESS AUDIT")
print("="*70)
# real NaNs
na_counts = df.isna().sum()
print("Native NaN counts:\n", na_counts[na_counts > 0])


2. MISSINGNESS AUDIT
Native NaN counts:
 transaction_id        4
customer_id           4
customer_name         4
email                32
transaction_date    139
product_category      4
quantity              4
unit_price           14
discount_pct          4
payment_method        4
country               4
shipping_weight       4
rating               76
is_returned          21
notes               245
dtype: int64


## placeholder missing values hiding inside object columns

In [5]:
placeholders = ["N/A", "n/a", "unknown", "-", "?", "", " ", "  "]
for col in df.select_dtypes(include=["object", "string"]).columns:
    hidden = df[col].isin(placeholders).sum()
    if hidden:
        issues.append((col, f"placeholder-missing values ({placeholders})", hidden))
        print(f"{col}: {hidden} placeholder-missing values")

unit_price: 30 placeholder-missing values
notes: 109 placeholder-missing values


In [6]:
empty_rows = df.isna().all(axis=1).sum()
if empty_rows:
    issues.append(("__row__", "fully empty rows", empty_rows))
    print(f"Fully empty rows: {empty_rows}")

Fully empty rows: 4


In [7]:
print("\n" + "="*70)
print("3. DUPLICATE AUDIT")
print("="*70)
exact_dupes = df.duplicated().sum()
print("Exact duplicate rows:", exact_dupes)
issues.append(("__row__", "exact duplicate rows", exact_dupes))
 
id_dupes = df["transaction_id"].duplicated().sum()
print("Duplicate transaction_id values (should be unique key!):", id_dupes)
issues.append(("transaction_id", "duplicate primary-key values", id_dupes))
 



3. DUPLICATE AUDIT
Exact duplicate rows: 18
Duplicate transaction_id values (should be unique key!): 28


## # unit_price: mixed currency strings + numeric

In [8]:
print("\n" + "="*70)
print("4. TYPE / FORMAT AUDIT")
print("="*70)
def parse_price(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.lower() in ["n/a", "unknown", "-", "?", ""]:
        return np.nan
    s = s.replace("$", "").replace(",", "")
    s = s.replace("PKR", "").strip()
    try:
        return float(s)
    except ValueError:
        return np.nan
df["unit_price_clean"] = df["unit_price"].apply(parse_price)
bad_price = df["unit_price_clean"].isna().sum() - df["unit_price"].isna().sum()
print(f"unit_price: mixed formats detected ($, PKR, commas, placeholders). "
      f"{df['unit_price'].apply(lambda x: isinstance(x,str) and any(c in str(x) for c in '$,PKR')).sum()} "
      f"rows had currency symbols/commas.")
issues.append(("unit_price", "inconsistent formatting: $ / PKR / commas / placeholder text mixed with numbers",
                df["unit_price"].apply(lambda x: isinstance(x, str)).sum()))


4. TYPE / FORMAT AUDIT
unit_price: mixed formats detected ($, PKR, commas, placeholders). 152 rows had currency symbols/commas.


## transaction_date: multiple formats

In [9]:
def try_parse_date(x):
    if pd.isna(x):
        return None
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d %b %Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT
 
df["transaction_date_parsed"] = df["transaction_date"].apply(try_parse_date)
unparsed_dates = df["transaction_date_parsed"].isna().sum() - df["transaction_date"].isna().sum()
print(f"transaction_date: at least 4 distinct date formats mixed in one column; "
      f"{df['transaction_date'].isna().sum()} missing outright.")
issues.append(("transaction_date", "at least 4 different date formats mixed in one column",
                len(df) - df["transaction_date"].isna().sum()))
 
future_dates = (df["transaction_date_parsed"] > pd.Timestamp.now()).sum()
print("Future-dated transactions (impossible):", future_dates)
issues.append(("transaction_date", "future / impossible dates", future_dates))
 

transaction_date: at least 4 distinct date formats mixed in one column; 139 missing outright.
Future-dated transactions (impossible): 24


## # categorical text inconsistency

In [10]:
print("\nproduct_category raw value counts (look for casing/typo variants):")
print(df["product_category"].value_counts())
issues.append(("product_category", "casing/whitespace/typo variants of the same category",
                df["product_category"].nunique() - 8))
 
print("\npayment_method raw value counts:")
print(df["payment_method"].value_counts())
issues.append(("payment_method", "casing/abbreviation variants of the same method",
                df["payment_method"].nunique() - 5))
 
print("\nis_returned raw value counts:")
print(df["is_returned"].value_counts(dropna=False))
issues.append(("is_returned", "inconsistent boolean encoding (Yes/No/Y/N/yes/no/NaN)",
                df["is_returned"].nunique()))


product_category raw value counts (look for casing/typo variants):
product_category
Toys                55
Grocery             45
Books               41
Sports              40
BEAUTY              38
Book                38
beauty              38
Groceries           38
groceries           37
sports              36
books               36
Clothing            35
Appearal            34
Toy                 33
Sport               32
Home & Kitchen      32
Home and Kitchen    32
toys                31
Beauty              31
Home&Kitchen        27
Electronic          26
Apparel             23
home & kitchen      22
electronics         18
Electronics         18
apparel             14
Electronics         13
ELECTRONICS         12
Name: count, dtype: int64

payment_method raw value counts:
payment_method
Bank_Transfer       76
wallet              66
e-wallet            61
Bank Transfer       60
Debit Card          56
bank transfer       56
credit card         55
Wallet              52
debit card  

## shipping_weight: mixed units

In [11]:
lbs_rows = df["shipping_weight"].astype(str).str.contains("lbs", na=False).sum()
print(f"\nshipping_weight: {lbs_rows} rows recorded in lbs instead of kg (unit inconsistency)")
issues.append(("shipping_weight", "mixed units: kg and lbs in the same column", lbs_rows))
 


shipping_weight: 103 rows recorded in lbs instead of kg (unit inconsistency)


In [12]:
print("\n" + "="*70)
print("5. RANGE / DOMAIN-VALIDITY AUDIT")
print("="*70)
 
qty_numeric = pd.to_numeric(df["quantity"], errors="coerce")
neg_qty = (qty_numeric < 0).sum()
extreme_qty = (qty_numeric > 100).sum()
print(f"quantity: {neg_qty} negative (impossible), {extreme_qty} extreme outliers (>100 units)")
issues.append(("quantity", "negative values (impossible)", neg_qty))
issues.append(("quantity", "extreme outliers (>100 units in one txn)", extreme_qty))
 
disc_numeric = pd.to_numeric(df["discount_pct"], errors="coerce")
bad_disc = ((disc_numeric > 100) | (disc_numeric < 0)).sum()
print(f"discount_pct: {bad_disc} values outside valid 0-100 range")
issues.append(("discount_pct", "values outside valid 0-100% range", bad_disc))
 
rating_numeric = pd.to_numeric(df["rating"], errors="coerce")
bad_rating = ((rating_numeric > 5) | (rating_numeric < 1)).sum()
print(f"rating: {bad_rating} values outside valid 1-5 range")
issues.append(("rating", "values outside valid 1-5 range", bad_rating))
 
weight_numeric = pd.to_numeric(
    df["shipping_weight"].astype(str).str.replace(" lbs", "", regex=False),
    errors="coerce"
)
neg_weight = (weight_numeric < 0).sum()
print(f"shipping_weight: {neg_weight} negative values (impossible)")
issues.append(("shipping_weight", "negative values (impossible)", neg_weight))
 
bad_email = ~df["email"].astype(str).str.match(r"^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$")
print(f"email: {bad_email.sum()} malformed / blank / missing '@' or domain")
issues.append(("email", "malformed emails (missing @, bad domain, or blank)", int(bad_email.sum())))


5. RANGE / DOMAIN-VALIDITY AUDIT
quantity: 22 negative (impossible), 19 extreme outliers (>100 units)
discount_pct: 43 values outside valid 0-100 range
rating: 24 values outside valid 1-5 range
shipping_weight: 11 negative values (impossible)
email: 114 malformed / blank / missing '@' or domain


In [13]:
print("\n" + "="*70)
print("6. REFERENTIAL INTEGRITY AUDIT")
print("="*70)
orphan_customers = ~df["customer_id"].isin(customers["customer_id"])
print(f"Transactions with customer_id NOT present in customers_reference.csv: {orphan_customers.sum()}")
issues.append(("customer_id", "orphan customer_id not present in customers table", int(orphan_customers.sum())))


6. REFERENTIAL INTEGRITY AUDIT
Transactions with customer_id NOT present in customers_reference.csv: 39


In [14]:
print("\n" + "="*70)
print("7. SUMMARY REPORT")
print("="*70)
report = pd.DataFrame(issues, columns=["column", "issue", "count"])
report = report[report["count"] > 0].sort_values("count", ascending=False)
print(report.to_string(index=False))
print(f"\nTotal distinct issue findings: {len(report)}")


7. SUMMARY REPORT
          column                                                                           issue  count
      unit_price inconsistent formatting: $ / PKR / commas / placeholder text mixed with numbers    865
transaction_date                           at least 4 different date formats mixed in one column    740
           email                              malformed emails (missing @, bad domain, or blank)    114
           notes placeholder-missing values (['N/A', 'n/a', 'unknown', '-', '?', '', ' ', '  '])    109
 shipping_weight                                      mixed units: kg and lbs in the same column    103
    discount_pct                                               values outside valid 0-100% range     43
     customer_id                               orphan customer_id not present in customers table     39
      unit_price placeholder-missing values (['N/A', 'n/a', 'unknown', '-', '?', '', ' ', '  '])     30
  transaction_id                             